<a href="https://colab.research.google.com/github/IT24104155/se3082-lab-10-2026/blob/main/notebooks/SE3082_Lab10_IT24104155.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap --format=csv


Exercise 1: Timing a GPU program

In [ ]:

%%writefile ex1_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    // Each block runs only one addition
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// Track CPU wall-clock time in seconds
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;             // Host copies of a, b, c
    int *d_a, *d_b, *d_c;       // Device copies of a, b, c
    int size = N * sizeof(int);
    double dStart, dElaps;

    // Allocate space for device copies
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate host copies and initialize input values
    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Start timer immediately before the kernel launch
    dStart = cpuSecond();

    // Launch add() with N blocks and one thread per block
    add<<<N, 1>>>(d_a, d_b, d_c);

    // Wait for the GPU to finish before stopping the timer
    cudaDeviceSynchronize();

    dElaps = cpuSecond() - dStart;

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // Print the first five results
    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    // Check the GPU results against the CPU
    int errors = 0;

    for (int r = 0; r < N; r++) {
        if (c[r] != a[r] + b[r])
            errors++;
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    // Cleanup
    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


In [ ]:

!nvcc -arch=sm_75 ex1_timed.cu -o ex1_timed && ./ex1_timed


Part b) Lab 9 activities timed

In [ ]:

%%writefile ex3_blocks_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Time the kernel only
    dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < N; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++) {
        if (c[r] != a[r] + b[r])
            errors++;
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


In [ ]:

!nvcc -arch=sm_75 ex3_blocks_timed.cu -o ex3_blocks_timed && ./ex3_blocks_timed


In [ ]:

%%writefile ex4_threads_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void addT(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Time the kernel only
    dStart = cpuSecond();
    addT<<<1, N>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < N; r++)
        printf("%d) %d + %d = %d\n", r, a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++) {
        if (c[r] != a[r] + b[r])
            errors++;
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


In [ ]:

!nvcc -arch=sm_75 ex4_threads_timed.cu -o ex4_threads_timed && ./ex4_threads_timed


In [ ]:

%%writefile ex5_vecmul_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 10000000
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n)
        c[i] = a[i] * b[i];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    size_t size = (size_t)N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    int blocks = (N + THREADS_PER_BLOCK - 1)
                 / THREADS_PER_BLOCK;

    printf("Launching %d blocks x %d threads\n",
           blocks, THREADS_PER_BLOCK);

    // Time the kernel only
    dStart = cpuSecond();
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);
    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int i = N - 1000; i < N; i++) {
        printf("%d) %d x %d = %d\n",
               i, a[i], b[i], c[i]);
    }

    int errors = 0;
    for (int i = 0; i < N; i++) {
        if (c[i] != a[i] * b[i])
            errors++;
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


In [ ]:

!nvcc -arch=sm_75 ex5_vecmul_timed.cu -o ex5_vecmul_timed && ./ex5_vecmul_timed


In [ ]:

%%writefile ex6_matrix_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define ROWS 10000
#define COLS 10000

__global__ void mulElem(int *a, int *b, int *c,
                        int rows, int cols) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < rows && col < cols) {
        size_t idx = (size_t)row * cols + col;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    size_t count = (size_t)ROWS * COLS;
    size_t size = count * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size);
    random_ints(a, count);

    b = (int *)malloc(size);
    random_ints(b, count);

    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(32, 16);

    dim3 numBlocks(
        (COLS + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y
    );

    printf("Grid: %d x %d blocks of %d x %d threads\n",
           numBlocks.x, numBlocks.y,
           threadsPerBlock.x, threadsPerBlock.y);

    // Time the kernel only
    dStart = cpuSecond();
    mulElem<<<numBlocks, threadsPerBlock>>>(
        d_a, d_b, d_c, ROWS, COLS
    );
    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // Print the last 1000 elements of the final row
    printf("Last 1000 matrix elements:\n");

    for (int col = COLS - 1000; col < COLS; col++) {
        size_t idx = (size_t)(ROWS - 1) * COLS + col;

        printf("C[%d][%d] = %d x %d = %d\n",
               ROWS - 1, col, a[idx], b[idx], c[idx]);
    }

    int errors = 0;
    for (size_t i = 0; i < count; i++) {
        if (c[i] != a[i] * b[i])
            errors++;
    }

    printf("Verification: %d mismatches out of %zu\n",
           errors, count);
    printf("Time taken - %.7f\n", dElaps);

    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


In [ ]:

!nvcc -arch=sm_75 ex6_matrix_timed.cu -o ex6_matrix_timed && ./ex6_matrix_timed


> Reference measurements from the original source author; not new runs for IT24104155.

### Exercise 1 (b) – Execution Time Measurements

| Lab 9 Exercise | Launch Configuration | Run 1 (s) | Run 2 (s) | Run 3 (s) |
|---|---|---:|---:|---:|
| Exercise 3 – Vector Addition | 512 blocks × 1 thread | 0.0001049 | 0.0001259 | 0.0001171 |
| Exercise 4 – Vector Addition | 1 block × 512 threads | 0.0001099 | 0.0001638 | 0.0001268 |
| Exercise 5 – Vector Multiplication | 19,532 blocks × 512 threads | 0.0006242 | 0.0006251 | 0.0006452 |
| Exercise 6 – Matrix Element-wise Multiplication | 313 × 625 blocks; 32 × 16 threads per block | 0.0050559 | 0.0050390 | 0.0050471 |

Part c)

In [ ]:

%%writefile ex1c_vecmul_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#include <cuda_runtime.h>

#define N 100000000
#define THREADS_PER_BLOCK 512

__global__ void multiply(int *a, int *b, int *c, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n)
    {
        c[i] = a[i] * b[i];
    }
}

void random_ints(int *x, int size)
{
    for (int i = 0; i < size; i++)
    {
        x[i] = rand() % 100;
    }
}

double cpuSecond()
{
    struct timeval tp;
    gettimeofday(&tp, NULL);

    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void)
{
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    size_t size = (size_t)N * sizeof(int);
    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;

    double start, kernelTime, totalTime;

    // Allocate host memory
    a = (int *)malloc(size);
    b = (int *)malloc(size);
    c = (int *)malloc(size);

    if (a == NULL || b == NULL || c == NULL)
    {
        printf("Host memory allocation failed.\n");
        free(a);
        free(b);
        free(c);
        return 1;
    }

    // Initialise input arrays
    random_ints(a, N);
    random_ints(b, N);

    // Allocate GPU memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Start total timer: transfers + kernel + result transfer
    start = cpuSecond();

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Measure only GPU kernel execution
    double kernelStart = cpuSecond();

    multiply<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);
    cudaDeviceSynchronize();

    kernelTime = cpuSecond() - kernelStart;

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    totalTime = cpuSecond() - start;

    // Display a few results
    for (int i = 0; i < 5; i++)
    {
        printf("%d * %d = %d\n", a[i], b[i], c[i]);
    }

    // Verify the result
    int errors = 0;

    for (int i = 0; i < N; i++)
    {
        if (c[i] != a[i] * b[i])
        {
            errors++;
        }
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Kernel execution time: %.7f seconds\n", kernelTime);
    printf("Total execution time:  %.7f seconds\n", totalTime);

    // Free host memory
    free(a);
    free(b);
    free(c);

    // Free GPU memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex1c_vecmul_timed.cu -o ex1c_vecmul_timed && ./ex1c_vecmul_timed

> Reference measurements from the original source author; not new runs for IT24104155.

### Exercise 1(c) – Kernel Time vs Total Execution Time

The vector multiplication program was modified to process 100,000,000 integer elements. The execution time of the CUDA kernel and the total time for transferring input data to the GPU, performing multiplication, and copying the results back to the CPU were measured.

| Measurement | Execution Time |
|---|---:|
| Kernel execution time | 0.0047121 seconds |
| Total execution time | 0.4391770 seconds |

**Verification:** 0 mismatches out of 100,000,000 elements.

**Observation:** The kernel execution time was significantly shorter than the total execution time. This demonstrates that memory transfers between the CPU and GPU contribute substantially to the overall execution time. Therefore, measuring only kernel execution time does not represent the full cost of running a GPU program.

Exercise 2: Catching errors with the CHECK() macro

In [ ]:

%%writefile ex2_check.cu
#include <stdio.h>
#include <cuda_runtime.h>

// Macro to check CUDA runtime errors
#define CHECK(call)                                                   \
do                                                                    \
{                                                                     \
    cudaError_t error = call;                                         \
    if (error != cudaSuccess)                                         \
    {                                                                 \
        fprintf(stderr, "CUDA Error at %s:%d - %s\n",                 \
                __FILE__, __LINE__, cudaGetErrorString(error));       \
        exit(EXIT_FAILURE);                                           \
    }                                                                 \
} while (0)

__global__ void add(int *a, int *b, int *c)
{
    int i = threadIdx.x;
    c[i] = a[i] + b[i];
}

int main(void)
{
    const int N = 5;
    int a[N] = {1, 2, 3, 4, 5};
    int b[N] = {10, 20, 30, 40, 50};
    int c[N];

    int *d_a, *d_b, *d_c;
    size_t size = N * sizeof(int);

    // Allocate GPU memory and check for errors
    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    // Copy input arrays to the GPU
    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    // Launch the kernel
    add<<<1, N>>>(d_a, d_b, d_c);

    // Check for kernel launch errors and execution errors
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    // Copy results back to the CPU
    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    // Display results
    for (int i = 0; i < N; i++)
    {
        printf("%d + %d = %d\n", a[i], b[i], c[i]);
    }

    // Release GPU memory
    CHECK(cudaFree(d_a));
    CHECK(cudaFree(d_b));
    CHECK(cudaFree(d_c));

    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex2_check.cu -o ex2_check && ./ex2_check

Exercise 3: Scaling a picture with a 2D grid

In [ ]:

%%writefile ex3_picture.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define WIDTH 800
#define HEIGHT 600
#define THREADS_PER_BLOCK 16

// Macro to check CUDA errors
#define CHECK(call)                                                   \
do                                                                    \
{                                                                     \
    cudaError_t error = call;                                         \
    if (error != cudaSuccess)                                         \
    {                                                                 \
        fprintf(stderr, "CUDA Error at %s:%d - %s\n",                 \
                __FILE__, __LINE__, cudaGetErrorString(error));       \
        exit(EXIT_FAILURE);                                           \
    }                                                                 \
} while (0)

// CUDA kernel to process picture pixels
__global__ void processPicture(unsigned char *picture,
                               int width, int height)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x < width && y < height)
    {
        int index = y * width + x;
        int value = picture[index] * 2;

        if (value > 255)
        {
            value = 255;
        }

        picture[index] = (unsigned char)value;
    }
}

int main(void)
{
    size_t size = (size_t)WIDTH * HEIGHT * sizeof(unsigned char);

    unsigned char *picture;
    unsigned char *d_picture;

    // Allocate host memory
    picture = (unsigned char *)malloc(size);

    if (picture == NULL)
    {
        printf("Host memory allocation failed.\n");
        return 1;
    }

    // Initialise the picture with sample pixel values
    for (int i = 0; i < WIDTH * HEIGHT; i++)
    {
        picture[i] = i % 256;
    }

    // Allocate GPU memory
    CHECK(cudaMalloc((void **)&d_picture, size));

    // Copy picture from CPU to GPU
    CHECK(cudaMemcpy(d_picture, picture, size,
                     cudaMemcpyHostToDevice));

    // Configure the 2D grid and blocks
    dim3 threadsPerBlock(16, 16);

    dim3 numBlocks(
        (WIDTH + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (HEIGHT + threadsPerBlock.y - 1) / threadsPerBlock.y
    );

    // Execute the CUDA kernel
    processPicture<<<numBlocks, threadsPerBlock>>>(
        d_picture, WIDTH, HEIGHT);

    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    // Copy processed picture back to CPU
    CHECK(cudaMemcpy(picture, d_picture, size,
                     cudaMemcpyDeviceToHost));

    // Display the first 10 processed pixel values
    printf("First 10 processed pixel values:\n");

    for (int i = 0; i < 10; i++)
    {
        printf("%u ", (unsigned int)picture[i]);
    }

    printf("\n");

    // Verify the processed pixels
    int errors = 0;

    for (int i = 0; i < WIDTH * HEIGHT; i++)
    {
        unsigned int original = i % 256;
        unsigned int expected = original * 2;

        if (expected > 255)
        {
            expected = 255;
        }

        if (picture[i] != expected)
        {
            errors++;
        }
    }

    printf("Verification: %d mismatches out of %d pixels\n",
           errors, WIDTH * HEIGHT);

    printf("Image dimensions: %d x %d\n", WIDTH, HEIGHT);
    printf("Blocks: %d x %d\n", numBlocks.x, numBlocks.y);
    printf("Threads per block: %d x %d\n",
           threadsPerBlock.x, threadsPerBlock.y);

    // Free memory
    CHECK(cudaFree(d_picture));
    free(picture);

    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex3_picture.cu -o ex3_picture && ./ex3_picture

Exercise 4 – Multiplying two 3D matrices element by element

In [ ]:

%%writefile ex4_matrix3d.cu
#include <stdio.h>
#include <stdlib.h>
#include <sys/time.h>
#include <cuda_runtime.h>

#define N 500
#define THREADS_PER_BLOCK 8

// Macro to check CUDA errors
#define CHECK(call)                                                   \
do                                                                    \
{                                                                     \
    cudaError_t error = call;                                         \
    if (error != cudaSuccess)                                         \
    {                                                                 \
        fprintf(stderr, "CUDA Error at %s:%d - %s\n",                 \
                __FILE__, __LINE__, cudaGetErrorString(error));       \
        exit(EXIT_FAILURE);                                           \
    }                                                                 \
} while (0)

// Return the current time in seconds
double cpuSecond()
{
    struct timeval tp;
    gettimeofday(&tp, NULL);

    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

// CUDA kernel for element-wise 3D matrix multiplication
__global__ void matrixMul3D(const int *a, const int *b, int *c, int n)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int z = blockIdx.z * blockDim.z + threadIdx.z;

    if (x < n && y < n && z < n)
    {
        int index = z * n * n + y * n + x;
        c[index] = a[index] * b[index];
    }
}

int main(void)
{
    size_t totalElements = (size_t)N * N * N;
    size_t size = totalElements * sizeof(int);

    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    // Allocate host memory
    a = (int *)malloc(size);
    b = (int *)malloc(size);
    c = (int *)malloc(size);

    if (a == NULL || b == NULL || c == NULL)
    {
        printf("Host memory allocation failed.\n");
        free(a);
        free(b);
        free(c);
        return 1;
    }

    // Initialise input matrices with small values
    for (size_t i = 0; i < totalElements; i++)
    {
        a[i] = (int)(i % 10);
        b[i] = (int)((i % 10) + 1);
    }

    // Allocate GPU memory
    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    // Copy input matrices to GPU
    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    // Configure a 3D grid and 3D blocks
    dim3 threadsPerBlock(THREADS_PER_BLOCK,
                         THREADS_PER_BLOCK,
                         THREADS_PER_BLOCK);

    dim3 numBlocks(
        (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK,
        (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK,
        (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK
    );

    // Measure kernel execution time
    double start = cpuSecond();

    matrixMul3D<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, N);

    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    double kernelTime = cpuSecond() - start;

    // Copy result back to host
    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    // Verify every element
    size_t errors = 0;

    for (size_t i = 0; i < totalElements; i++)
    {
        if (c[i] != a[i] * b[i])
        {
            errors++;
        }
    }

    printf("Matrix dimensions: %d x %d x %d\n", N, N, N);
    printf("Blocks: %d x %d x %d\n",
           numBlocks.x, numBlocks.y, numBlocks.z);
    printf("Threads per block: %d x %d x %d\n",
           threadsPerBlock.x,
           threadsPerBlock.y,
           threadsPerBlock.z);

    printf("Verification: %zu mismatches out of %zu elements\n",
           errors, totalElements);
    printf("Kernel execution time: %.7f seconds\n", kernelTime);

    // Display the first 10 results
    printf("First 10 results:\n");

    for (int i = 0; i < 10; i++)
    {
        printf("%d ", c[i]);
    }

    printf("\n");

    // Free memory
    CHECK(cudaFree(d_a));
    CHECK(cudaFree(d_b));
    CHECK(cudaFree(d_c));

    free(a);
    free(b);
    free(c);

    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex4_matrix3d.cu -o ex4_matrix3d && ./ex4_matrix3d